# 13 · Model Misspecification

*Measure bias caused by the assumptions that formal covariance treats as fixed.*

**Learning objectives**

- separate conditional precision from accuracy under model error
- design one-factor sensitivity experiments for geometry, medium, mesh, and noise
- identify coherent spatial residuals
- report bias beside formal uncertainty

**Prerequisites:** Chapters 08–09  
**Estimated time:** 45–60 minutes


## Motivation

Posterior covariance and resolution are computed under the assumed $G$ and
$C_d$. They cannot reveal that the fault dips differently, the Earth is
layered, the mesh aliases the source, or atmospheric noise was treated as
white. Sensitivity analysis changes those fixed assumptions and measures what
the formal errors omit.


## Bias from an incorrect forward operator

Suppose truth uses $G_t$ while inversion uses $G_a$. Even with zero-mean noise,

$$E[\hat m]-m_t = G_a^{-g}G_t m_t-m_t.$$

This deterministic bias is absent from the conditional covariance computed
with $G_a$. Residuals may expose it when the wrong model cannot reproduce the
data, but compensating slip can also hide it. Precision is repeatability under
the assumed model; accuracy requires confronting model-form error.


## Experiment: invert the right data with the wrong dip

Only dip changes below. The same regularization, stations, covariance, and
seed are used so the resulting difference can be attributed cleanly.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
geometry = dict(
    lat=34.0, lon=-118.0, depth=10_000.0, strike=90.0,
    length=36_000.0, width=18_000.0, n_length=6, n_width=3,
)
truth_fault = geodef.Fault.planar(dip=25.0, **geometry)
wrong_fault = geodef.Fault.planar(dip=45.0, **geometry)
i = np.arange(truth_fault.n_patches) % 6
j = np.arange(truth_fault.n_patches) // 6
true_slip = 1.2 * np.exp(-((i - 2.5) / 1.5) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.25, -117.75, 8), np.linspace(33.83, 34.18, 6)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = truth_fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.003
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="synthetic_gnss",
)
kwargs = dict(
    datasets=gnss, components="dip", regularization="laplacian",
    regularization_strength=1.0, bounds=(0.0, None),
)
right = geodef.solve(truth_fault, **kwargs)
wrong = geodef.solve(wrong_fault, **kwargs)
formal_sigma = geodef.invert.model_uncertainty(wrong, wrong_fault, gnss)
bias = wrong.dip_slip - true_slip
print(f"right/wrong reduced chi2: {right.reduced_chi2:.2f}, {wrong.reduced_chi2:.2f}")
print(f"largest |bias| / formal sigma: {np.max(np.abs(bias) / formal_sigma):.1f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
limit = max(true_slip.max(), right.dip_slip.max(), wrong.dip_slip.max())
for ax, field, title in [
    (axes[0], true_slip, "Truth: dip 25°"),
    (axes[1], right.dip_slip, "Recovered: dip 25°"),
    (axes[2], wrong.dip_slip, "Recovered: assumed dip 45°"),
]:
    geodef.plot.slip(truth_fault if ax is not axes[2] else wrong_fault, field, ax=ax, vmin=0, vmax=limit, title=title, colorbar_label="Slip (m)")

residual_up = wrong.residuals[2::3]
fig2, ax2 = plt.subplots(figsize=(6, 4), constrained_layout=True)
points = ax2.scatter(station_lon, station_lat, c=residual_up * 1e3, cmap="RdBu_r")
ax2.set(xlabel="Longitude (°)", ylabel="Latitude (°)", title="Wrong-geometry vertical residual")
fig2.colorbar(points, ax=ax2, label="Residual (mm)");


The wrong geometry can produce coherent bias several formal standard
deviations large. Reduced chi-squared may worsen, but compensating slip often
keeps it deceptively plausible. The spatial residual map carries information a
single statistic discards.

Repeat the same experiment for Poisson's ratio, patch scale, and correlated
noise. Change one assumption at a time, use identical noise realizations, and
report quantities of interest such as peak slip and $M_w$.


## Checkpoint questions

**Why is the formal sigma unchanged by uncertainty in dip?**
<details><summary>Answer</summary>The linear covariance conditions on the
chosen dip; dip is not a random parameter in that calculation.</details>

**Can unstructured residuals prove the model is correct?**
<details><summary>Answer</summary>No. Some errors are absorbed by slip or hidden
below noise.</details>

**What makes a sensitivity experiment interpretable?**
<details><summary>Answer</summary>Change one declared assumption while holding
data, seed, and other analysis choices fixed.</details>


## Common mistakes

- Calling conditional error bars “total uncertainty.”
- Changing geometry and regularization simultaneously, then attributing the bias.
- Looking only at scalar RMS instead of spatial/component residual structure.
- Selecting only sensitivity cases that leave the preferred interpretation intact.


## Recap

- Formal covariance quantifies uncertainty inside the assumed model family.
- Wrong geometry or physics can create bias larger than formal sigma.
- Residual forensics and sensitivity tables belong beside the preferred model.

Use the assessment guide in [workflow](../docs/workflow.md).


## Exercises

Worked solutions are in `solutions/13_model_misspecification_solutions.ipynb`.

1. Sweep assumed dip and plot moment-magnitude bias.
2. Generate correlated noise but invert with diagonal covariance.
3. Coarsen the inversion mesh and locate aliased slip.
4. Challenge: redesign station locations to make a 5 km depth error easiest to detect.


## Further reading

- Tarantola (2005), inverse problems and model assumptions.
- Box (1976), the usefulness and wrongness of models.
- Segall (2010), elastic-model limitations in geodesy.
